# 07 - 2D Ito Differential Equations and Visualization

This notebook explores 2D stochastic differential equations (SDEs) in the Ito formulation and their application to SGD dynamics.

**Converted from:** `2d_ito_diff_graph.nb` (Mathematica)

## Contents:
1. Ito SDE formulation for SGD
2. Phase space visualization
3. Stochastic trajectory simulation
4. Drift and diffusion analysis
5. Comparison with deterministic dynamics

## Background

SGD in continuous time can be modeled as an Ito SDE:

$$d\theta_t = b(\theta_t) dt + \sigma(\theta_t) dW_t$$

where $b(\theta)$ is the drift (negative gradient) and $\sigma(\theta)$ is the diffusion coefficient.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from scipy.integrate import odeint

# Add utils to path
sys.path.insert(0, str(Path.cwd().parent / 'utils'))

from sgd_simulator import SGDSimulator, SGDConfig
from loss_functions import SmoothNonlinearLoss, generate_noisy_data
from visualization import plot_loss_landscape, plot_trajectories

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

## 1. Ito SDE Formulation

For a 2D parameter space $\theta = (\theta_1, \theta_2)$, the Ito SDE is:

$$d\theta_1 = -\frac{\partial L}{\partial \theta_1} dt + \sqrt{2D} dW_1$$
$$d\theta_2 = -\frac{\partial L}{\partial \theta_2} dt + \sqrt{2D} dW_2$$

where:
- $L(\theta_1, \theta_2)$ is the loss function
- $D$ is the diffusion coefficient (related to learning rate and gradient noise)
- $dW_1, dW_2$ are independent Wiener increments

In [ ]:
# Generate data and loss function
np.random.seed(42)

x_data, y_data = generate_noisy_data(
    x_range=(-3, 3),
    n_points=20,
    n_samples_per_point=5,
    noise_std=0.5,
    p=1.0,
    random_state=42
)

loss_obj = SmoothNonlinearLoss(p=1.0)

print(f"Loss function created with {len(x_data)} data points")

In [ ]:
def euler_maruyama_2d(drift_fn, diffusion_coef, initial_state, dt, n_steps, random_state=None):
    """
    Euler-Maruyama method for 2D Ito SDE.
    
    dX = drift(X) * dt + sqrt(2*D) * dW
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    trajectory = np.zeros((n_steps + 1, 2))
    trajectory[0] = initial_state
    
    sqrt_dt = np.sqrt(dt)
    sqrt_2D = np.sqrt(2 * diffusion_coef)
    
    for i in range(n_steps):
        drift = drift_fn(trajectory[i])
        dW = np.random.randn(2) * sqrt_dt
        trajectory[i + 1] = trajectory[i] + drift * dt + sqrt_2D * dW
    
    return trajectory

# Define drift function (negative gradient)
def drift_function(params):
    gradient = loss_obj.gradient(params, x_data, y_data)
    return -gradient

print("Euler-Maruyama integrator defined")

## 2. Phase Space Visualization

Let's visualize the phase space by showing:
1. The drift field (negative gradient)
2. Sample stochastic trajectories
3. Comparison with deterministic gradient flow

In [ ]:
# Create vector field for drift
param_range = ((-1, 3), (-1, 3))
(a_min, a_max), (b_min, b_max) = param_range

n_grid = 20
a_vals = np.linspace(a_min, a_max, n_grid)
b_vals = np.linspace(b_min, b_max, n_grid)
A, B = np.meshgrid(a_vals, b_vals)

# Compute drift at each grid point
drift_a = np.zeros_like(A)
drift_b = np.zeros_like(B)

for i in range(n_grid):
    for j in range(n_grid):
        params = np.array([A[i, j], B[i, j]])
        drift = drift_function(params)
        drift_a[i, j] = drift[0]
        drift_b[i, j] = drift[1]

print("Drift vector field computed")

In [ ]:
# Plot drift vector field on loss landscape
fig, ax = plt.subplots(figsize=(14, 11))

# Background: loss landscape
n_contour = 100
a_fine = np.linspace(a_min, a_max, n_contour)
b_fine = np.linspace(b_min, b_max, n_contour)
A_fine, B_fine = np.meshgrid(a_fine, b_fine)

L = np.zeros_like(A_fine)
for i in range(n_contour):
    for j in range(n_contour):
        params = np.array([A_fine[i, j], B_fine[i, j]])
        L[i, j] = loss_obj(params, x_data, y_data)

contourf = ax.contourf(A_fine, B_fine, L, levels=30, cmap='viridis', alpha=0.6)
ax.contour(A_fine, B_fine, L, levels=30, colors='black', alpha=0.3, linewidths=0.5)
plt.colorbar(contourf, ax=ax, label='Loss')

# Overlay: drift vector field
ax.quiver(A, B, drift_a, drift_b, 
         scale=np.max(np.sqrt(drift_a**2 + drift_b**2)) * 15,
         color='white', alpha=0.8, width=0.004)

ax.set_xlabel('Parameter a', fontsize=12)
ax.set_ylabel('Parameter b', fontsize=12)
ax.set_title('Drift Vector Field (Negative Gradient) on Loss Landscape', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Arrows point in the direction of steepest descent (drift)")

## 3. Stochastic Trajectory Simulation

Now we'll simulate multiple stochastic trajectories using the Euler-Maruyama method and compare them with deterministic gradient descent.

In [ ]:
# Simulation parameters
diffusion_coef = 0.01  # D in the SDE
dt = 0.01  # Time step
n_steps = 1000  # Number of steps
initial_state = np.array([0.5, 2.5])

# Simulate multiple stochastic trajectories
n_trajectories = 5
stochastic_trajectories = []

for i in range(n_trajectories):
    traj = euler_maruyama_2d(
        drift_fn=drift_function,
        diffusion_coef=diffusion_coef,
        initial_state=initial_state,
        dt=dt,
        n_steps=n_steps,
        random_state=42 + i
    )
    stochastic_trajectories.append(traj)

print(f"Simulated {n_trajectories} stochastic trajectories")
print(f"Each trajectory has {n_steps + 1} points")

In [ ]:
# Deterministic trajectory (gradient descent)
def deterministic_ode(state, t):
    return drift_function(state)

t_eval = np.linspace(0, n_steps * dt, n_steps + 1)
deterministic_traj = odeint(deterministic_ode, initial_state, t_eval)

print("Deterministic trajectory computed")

In [ ]:
# Plot stochastic and deterministic trajectories
fig, ax = plt.subplots(figsize=(14, 11))

# Background: loss landscape
contourf = ax.contourf(A_fine, B_fine, L, levels=30, cmap='viridis', alpha=0.6)
ax.contour(A_fine, B_fine, L, levels=30, colors='black', alpha=0.3, linewidths=0.5)
plt.colorbar(contourf, ax=ax, label='Loss')

# Plot stochastic trajectories
colors = plt.cm.Reds(np.linspace(0.4, 0.9, n_trajectories))
for i, traj in enumerate(stochastic_trajectories):
    ax.plot(traj[:, 0], traj[:, 1], '-', linewidth=2, color=colors[i], 
           alpha=0.7, label=f'Stochastic {i+1}' if i < 3 else '')
    ax.plot(traj[0, 0], traj[0, 1], 'o', markersize=10, color=colors[i],
           markeredgecolor='black', markeredgewidth=2)

# Plot deterministic trajectory
ax.plot(deterministic_traj[:, 0], deterministic_traj[:, 1], 'b-', 
       linewidth=3, alpha=0.9, label='Deterministic')
ax.plot(deterministic_traj[0, 0], deterministic_traj[0, 1], 'bo', 
       markersize=12, markeredgecolor='black', markeredgewidth=2)

ax.set_xlabel('Parameter a', fontsize=12)
ax.set_ylabel('Parameter b', fontsize=12)
ax.set_title('Stochastic vs Deterministic Trajectories', fontsize=14)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Stochastic trajectories (red) show noise-induced exploration")
print("Deterministic trajectory (blue) follows exact gradient descent")

## 4. Drift and Diffusion Analysis

Let's analyze how drift and diffusion affect trajectory behavior by varying the diffusion coefficient.

In [ ]:
# Simulate with different diffusion coefficients
diffusion_levels = [0.001, 0.01, 0.05]
trajectories_by_diffusion = {}

for D in diffusion_levels:
    traj = euler_maruyama_2d(
        drift_fn=drift_function,
        diffusion_coef=D,
        initial_state=initial_state,
        dt=dt,
        n_steps=n_steps,
        random_state=42
    )
    trajectories_by_diffusion[D] = traj

print(f"Simulated trajectories for D = {diffusion_levels}")

In [ ]:
# Plot trajectories with different diffusion
fig, ax = plt.subplots(figsize=(14, 11))

# Background
contourf = ax.contourf(A_fine, B_fine, L, levels=30, cmap='viridis', alpha=0.6)
ax.contour(A_fine, B_fine, L, levels=30, colors='black', alpha=0.3, linewidths=0.5)
plt.colorbar(contourf, ax=ax, label='Loss')

# Plot trajectories
colors_diff = ['green', 'orange', 'red']
for i, (D, traj) in enumerate(trajectories_by_diffusion.items()):
    ax.plot(traj[:, 0], traj[:, 1], '-', linewidth=2.5, 
           color=colors_diff[i], alpha=0.8, label=f'D = {D}')
    ax.plot(traj[0, 0], traj[0, 1], 'o', markersize=10, 
           color=colors_diff[i], markeredgecolor='black', markeredgewidth=2)

ax.set_xlabel('Parameter a', fontsize=12)
ax.set_ylabel('Parameter b', fontsize=12)
ax.set_title('Effect of Diffusion Coefficient on Trajectories', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Higher diffusion leads to more exploration and wandering")

## 5. Time Evolution of Parameters

Let's examine how the parameters evolve over time for different diffusion levels.

In [ ]:
# Plot parameter evolution over time
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
time = np.arange(n_steps + 1) * dt

# Parameter a
for i, (D, traj) in enumerate(trajectories_by_diffusion.items()):
    axes[0].plot(time, traj[:, 0], '-', linewidth=2, 
                color=colors_diff[i], alpha=0.8, label=f'D = {D}')

axes[0].plot(time, deterministic_traj[:, 0], 'k--', linewidth=2, 
            alpha=0.6, label='Deterministic')
axes[0].set_xlabel('Time', fontsize=12)
axes[0].set_ylabel('Parameter a', fontsize=12)
axes[0].set_title('Parameter a Evolution', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Parameter b
for i, (D, traj) in enumerate(trajectories_by_diffusion.items()):
    axes[1].plot(time, traj[:, 1], '-', linewidth=2, 
                color=colors_diff[i], alpha=0.8, label=f'D = {D}')

axes[1].plot(time, deterministic_traj[:, 1], 'k--', linewidth=2, 
            alpha=0.6, label='Deterministic')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Parameter b', fontsize=12)
axes[1].set_title('Parameter b Evolution', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

In this notebook, we explored:

1. **Ito SDE Formulation**: Represented SGD as a continuous-time stochastic differential equation
2. **Phase Space Visualization**: Visualized the drift field (negative gradient) on the loss landscape
3. **Stochastic Trajectories**: Simulated SGD using the Euler-Maruyama method
4. **Drift vs Diffusion**: Compared stochastic trajectories with deterministic gradient descent
5. **Diffusion Analysis**: Examined how varying the diffusion coefficient affects exploration

**Key Insights:**
- Stochastic noise enables exploration beyond deterministic gradient descent
- Higher diffusion coefficients lead to more wandering and broader exploration
- The drift field points toward local minima, but diffusion can help escape
- Multiple stochastic trajectories from the same initial condition diverge due to noise

**Next Steps:**
- Perform comprehensive trajectory simulations with parameter sweeps
- Analyze statistical properties of SGD trajectories